# 03 실전 반도체 공정 데이터 분석 · 10. 보강 실습

- 강의 페이지: `Web/강좌/03_실전_반도체_공정_데이터분석/실전_반도체_공정_데이터분석_강의자료.html` → 목차 **보강 실습**
- 기준 모델·혼동행렬·임계값·모델 비교·놓친 불량을 직접 계산합니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

## 초보자 필수 보강 실습

모델을 만드는 것보다 중요한 것은 **무엇을 맞혀야 하는지**, **어떤 실수를 줄여야 하는지**를
이해하는 일입니다. 반도체 불량 탐지에서는 불량을 정상으로 놓치는 `False Negative`와
불량 재현율(`Recall`)을 특히 주의해서 봅니다.


### 준비 · 1~8단계 코드 실행 (모델 학습까지)

앞 단계 코드를 그대로 모아 한 번에 실행합니다. 출력은 앞 노트북과 같습니다.

In [ ]:
# ── 1단계 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 그래프 한글 설정 (Mac은 'AppleGothic', Colab·리눅스는 'NanumGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 2단계 ──
df = pd.read_csv('fab.csv')

# ── 3단계 결측값 처리 ──
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

# ── 4단계 분산 0 제거 ──
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")

# ── 5단계 특성 선택 ──
# 타겟을 0/1로 변환: 불량(1)을 양성 클래스로
y = (df['Pass_Fail'] == 1).astype(int)
X_all = df[numeric_cols].copy()

# 상위 K=20 개 센서 자동 선택
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

f_scores   = pd.Series(selector.scores_, index=numeric_cols)\
               .replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 선택된 상위 {K}개 센서:")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col}  F={f_scores[col]:6.1f}")

# ── 7단계 분리 & 스케일링 ──
X = df[top_k_cols].copy()    # 상위 K개 센서만!
# y = (df['Pass_Fail'] == 1).astype(int)  # 위에서 이미 만듦

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ⭐ 불균형 비율 유지 필수!
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"학습: {X_train.shape}, 그 중 불량 {y_train.sum()}건")
print(f"테스트: {X_test.shape}, 그 중 불량 {y_test.sum()}건")

# ── 8단계 모델 학습 ──
# 1) 로지스틱 회귀
lr_model = LogisticRegression(
    random_state=42, max_iter=1000,
    class_weight='balanced'   # ⭐ 소수 클래스에 가중치
)
lr_model.fit(X_train_scaled, y_train)

# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

# 예측
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)


### 실습 1 · 클래스 불균형과 기준 모델 이해하기

**목표:** 항상 정상이라고 예측하는 단순 모델과 비교해 정확도의 함정을 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: y_test의 정상/불량 건수와 비율을 출력하세요.
# TODO 2: 항상 정상(0)만 예측하는 배열을 만드세요.
# TODO 3: 이 기준 모델의 정확도와 불량 Recall을 계산하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`value_counts(normalize=True)`와 `accuracy_score`, `recall_score`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import accuracy_score, recall_score

print('테스트 판정 건수:')
print(y_test.value_counts().sort_index())
print('\n테스트 판정 비율(%):')
print(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))

always_normal = np.zeros(len(y_test), dtype=int)
baseline_accuracy = accuracy_score(y_test, always_normal)
baseline_recall = recall_score(y_test, always_normal, zero_division=0)
print(f'항상 정상 모델 정확도: {baseline_accuracy:.3f}')
print(f'항상 정상 모델 불량 Recall: {baseline_recall:.3f}')
print('정확도가 높아도 불량 Recall이 0이면 불량 탐지 모델로 쓸 수 없습니다.')


### 실습 2 · 혼동행렬을 숫자로 읽기

**목표:** 정상/불량 예측의 네 가지 경우와 놓친 불량 수를 직접 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 랜덤 포레스트 예측 결과로 혼동행렬을 만드세요.
# TODO 2: tn, fp, fn, tp 네 값으로 나누어 저장하세요.
# TODO 3: 불량 Recall = tp / (tp + fn)을 직접 계산하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

2x2 혼동행렬에는 `ravel()`을 적용할 수 있습니다. 분모가 0인지도 확인하세요.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
rf_predictions = rf_model.predict(X_test_scaled)
tn, fp, fn, tp = confusion_matrix(y_test, rf_predictions).ravel()
manual_recall = tp / (tp + fn) if (tp + fn) else 0

print(f'TN(정상을 정상): {tn}')
print(f'FP(정상을 불량): {fp}')
print(f'FN(불량을 정상으로 놓침): {fn}')
print(f'TP(불량을 불량): {tp}')
print(f'불량 Recall: {manual_recall:.3f}')


### 실습 3 · 예측 임계값과 Recall의 관계

**목표:** 불량 판정 기준을 바꿀 때 Recall과 Precision이 어떻게 달라지는지 비교합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 랜덤 포레스트의 불량 확률을 구하세요.
# TODO 2: 임계값 0.3, 0.5, 0.7마다 예측값을 만드세요.
# TODO 3: 각 임계값의 Precision과 Recall을 표로 정리하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

확률이 임계값 이상이면 1로 바꾸고 `precision_score`, `recall_score`를 계산합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import precision_score, recall_score

rf_probabilities = rf_model.predict_proba(X_test_scaled)[:, 1]
threshold_rows = []
for threshold in [0.3, 0.5, 0.7]:
    threshold_prediction = (rf_probabilities >= threshold).astype(int)
    threshold_rows.append({
        '임계값': threshold,
        'Precision': precision_score(y_test, threshold_prediction, zero_division=0),
        'Recall': recall_score(y_test, threshold_prediction, zero_division=0),
        '불량예측수': int(threshold_prediction.sum())
    })

threshold_table = pd.DataFrame(threshold_rows).round(3)
threshold_table


### 실습 4 · 두 모델을 같은 기준으로 비교하기

**목표:** 정확도만 보지 않고 불량 Precision, Recall, F1을 한 표에서 비교합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 로지스틱 회귀와 랜덤 포레스트의 예측값을 만드세요.
# TODO 2: 각 모델의 Accuracy, Precision, Recall, F1을 계산하세요.
# TODO 3: Recall이 높은 순서로 정렬하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

여러 모델을 리스트에 담아 반복하면 같은 계산 기준을 적용할 수 있습니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison_rows = []
for model_name, model in [
    ('로지스틱 회귀', lr_model),
    ('랜덤 포레스트', rf_model)
]:
    prediction = model.predict(X_test_scaled)
    comparison_rows.append({
        '모델': model_name,
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision_불량': precision_score(y_test, prediction, zero_division=0),
        'Recall_불량': recall_score(y_test, prediction, zero_division=0),
        'F1_불량': f1_score(y_test, prediction, zero_division=0)
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values('Recall_불량', ascending=False)
    .round(3)
)
model_comparison


### 실습 5 · 놓친 불량 사례 살펴보기

**목표:** False Negative 행을 찾아 어떤 센서값을 추가로 점검할지 준비합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: y_test가 1이면서 랜덤 포레스트 예측이 0인 위치를 찾으세요.
# TODO 2: 원래 X_test에서 해당 행을 선택하세요.
# TODO 3: 놓친 불량 수와 센서값 일부를 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`(y_test == 1) & (예측 == 0)` 조건을 만들고 불리언 배열의 위치로 `X_test`를 선택합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
false_negative_mask = (y_test.to_numpy() == 1) & (rf_predictions == 0)
false_negative_cases = X_test.loc[false_negative_mask].copy()

print(f'랜덤 포레스트가 놓친 불량: {len(false_negative_cases)}건')
display_cols = top_k_cols[:5]
if len(false_negative_cases) > 0:
    display(false_negative_cases[display_cols].head())
    print('이 행들은 원인 확정 대상이 아니라 추가 공정 점검 대상입니다.')
else:
    print('현재 테스트 데이터에서는 놓친 불량이 없습니다.')


### 꼭 알아둘 데이터 누수

실제 프로젝트에서는 **데이터를 학습용과 평가용으로 먼저 분리한 뒤** 다음 작업을
학습 데이터에만 맞춰야 합니다.

- 결측값을 채울 중앙값 계산
- 중요한 센서 선택
- 표준화 평균과 표준편차 계산

평가 데이터의 정보를 미리 사용하면 모델 성능이 실제보다 좋아 보일 수 있습니다.
AI에게 프로그램을 요청할 때도 이 순서를 명확히 적어 주는 것이 좋습니다.


## 마무리

- 정확도 0.933인 '항상 정상' 모델의 불량 Recall은 0입니다.
- 임계값을 낮추면 불량 Recall이 오르지만 오탐(FP)도 늘어납니다. 임계값은 테스트가 아닌 **별도 검증 데이터**에서 정하세요.

다음 노트북(`06_AI_미니프로젝트.ipynb`)에서 데이터 누수를 막은 버전을 AI와 함께 만들어 봅니다.